In [ ]:
%pip install -q fastapi uvicorn pyngrok nest-asyncio

!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [100 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,820 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa

In [ ]:
!fuser -k 11434/tcp
!pkill -f ollama
!ollama serve > ollama.log 2>&1 &

In [ ]:
!ollama pull qwen3-vl:8b-instruct

In [ ]:
!ollama list

In [ ]:
import json
from logging import exception
import asyncio
import nest_asyncio
from typing import List, Union, Dict, Any, Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import httpx

nest_asyncio.apply()

class ChatMessage(BaseModel):
  role: str
  content: str
  images: List[str] | None = None

class LLMPayload(BaseModel):
  model: str
  messages: List[ChatMessage]
  format: Optional[dict[str, Any]] = None
  temperature: float = 0.0
  think: bool = False
  stream: bool = True
  options: dict[str, Any] = {}

app = FastAPI(title="Medical AI Inference Gateway")

OLLAMA_API_URL = "http://127.0.0.1:11434/v1/chat/completions"

timeout_config = httpx.Timeout(300.0, connect=60.0, read=300.0, write=60.0)
limits_config = httpx.Limits(max_connections=50, max_keepalive_connections=10)

@app.post("/v1/image/completions")
async def process_vlm_request(payload: LLMPayload):
  """의료 메타데이터 및 Base64 이미지 페이로드를 받아 내부 로드된 VLM 모델로 추론을 수행하는 핵심 엔드포인트 입니다."""

  try:
    ollama_payload = {
        "model": payload.model,
        "messages": payload.model_dump()["messages"],
        "temperature": payload.temperature,
        "stream": True,
        "think": payload.think,
        "max_tokens": 2048,
        "options": payload.options
    }

    if payload.format is not None:
      ollama_payload["format"] = payload.format

    async def response_generator():
      async with httpx.AsyncClient(timeout=timeout_config, limits=limits_config) as client:
          try:
              async with client.stream("POST", OLLAMA_API_URL, json=ollama_payload) as response:
                  if response.status_code != 200:
                      yield f"data: {json.dumps({'error': 'Ollama 서버 응답 에러'})}\n\n"
                      return

                  async for line in response.aiter_lines():
                      if line:
                          yield f"{line}\n\n"

          except httpx.RemoteProtocolError as proto_err:
              print(f"[Gateway Error] 내부 Ollama 서버가 연산 중 연결을 강제 종료했습니다 (VRAM 초과 의심): {proto_err}")
              error_json = {
                  "choices": [{
                      "delta": {"content": "\n\n⚠️ **[프록시 게이트웨이 시스템 안내]** 로컬 AI 엔진 서버의 GPU 리소스(VRAM OOM) 부족으로 인해 분석 도중 연결이 유실되었습니다. 컨텍스트 크기를 줄이거나 하드웨어 상태를 점검하세요."}
                  }]
              }
              yield f"data: {json.dumps(error_json)}\n\n"
              yield "data: [DONE]\n\n"
          except Exception as stream_err:
              print(f"[Gateway Stream Exception]: {stream_err}")
              return

    return StreamingResponse(response_generator(), media_type="text/event-stream")
  except Exception as e:
    raise HTTPException(status_code=500, detail=f"Internal Server Error: {str(e)}")

NGROK_TOKEN = "3EzUV00bhdXtnBDjZ6hMUxdpIsv_7sYr1oc1Kf28T1JJrg6Yz"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000, domain="overrate-comprised-outfield.ngrok-free.dev")
print(f"\n 외부 프로세스에서 접근 가능한 API 주소-vlm: {public_url.public_url}/v1/image/completions\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="debug")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.create_task(server.serve())